# 🌏 การสร้างแผนที่ 3 มิติ ด้วย Python
**GE234 — Basic Programming for Geographers**  
Dept. of Geography, Thammasat University

---

## 📋 เนื้อหา Workshop
| Level | Library | เรื่อง |
|-------|---------|--------|
| 1 | `matplotlib` | 3D Surface plot จาก DEM (static) |
| 2 | `plotly` | Interactive 3D Surface (หมุนได้) |
| 3 | `pydeck` | Geospatial 3D บนแผนที่จริง |

**ข้อมูลที่ใช้**: SRTM 30m DEM ภาคเหนือไทย (เชียงใหม่–น่าน)

## ⚙️ 0 — Setup: ติดตั้ง Libraries

In [ ]:
# ติดตั้ง libraries ที่ไม่มีใน Colab by default
!pip install rasterio plotly pydeck earthengine-api --quiet
print("✅ ติดตั้งเสร็จ")

In [ ]:
# Import libraries ทั้งหมดที่จะใช้
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import pydeck as pdk
import pandas as pd
from scipy.ndimage import zoom
import warnings
warnings.filterwarnings('ignore')

print("✅ Import เสร็จทั้งหมด")

## 📦 เตรียมข้อมูล DEM

เราจะใช้ **2 วิธี** ในการได้ข้อมูล DEM:
1. **สร้าง Synthetic DEM** (สำหรับ Demo — ไม่ต้อง download)
2. **Download จาก OpenTopography** (ข้อมูลจริง — ต้องมี internet)

เริ่มจากวิธีที่ 1 ก่อน แล้วค่อยลองวิธีที่ 2

In [ ]:
# ─────────────────────────────────────────────────────────────
# วิธีที่ 1: สร้าง Synthetic DEM จำลองภาคเหนือไทย
# ลักษณะ: หุบเขาล้อมด้วยเทือกเขา คล้าย terrain จริง
# ─────────────────────────────────────────────────────────────
def create_synthetic_dem(size=300):
    """สร้าง DEM จำลองที่มีลักษณะคล้าย terrain ภาคเหนือไทย"""
    x = np.linspace(0, 4*np.pi, size)
    y = np.linspace(0, 4*np.pi, size)
    X, Y = np.meshgrid(x, y)

    # Base: gentle hills
    Z = (np.sin(X*0.5) * np.cos(Y*0.5) * 800
         + np.sin(X*1.2 + 0.5) * np.cos(Y*0.8) * 400
         + np.sin(X*2.5) * np.sin(Y*2.0) * 200)

    # เพิ่ม high peaks (คล้ายดอย)
    cx1, cy1 = size//4, size//3
    cx2, cy2 = 2*size//3, 2*size//3
    r1 = np.sqrt((np.arange(size)[:, None] - cx1)**2 + (np.arange(size)[None, :] - cy1)**2)
    r2 = np.sqrt((np.arange(size)[:, None] - cx2)**2 + (np.arange(size)[None, :] - cy2)**2)
    Z += 1200 * np.exp(-r1**2 / (2 * (size//10)**2))
    Z += 900 * np.exp(-r2**2 / (2 * (size//12)**2))

    # เพิ่ม noise
    np.random.seed(42)
    Z += np.random.normal(0, 30, Z.shape)

    # ทำให้ min = 200 m (เหมือน valley floor ในเชียงใหม่)
    Z = Z - Z.min() + 200

    return Z


# สร้าง DEM
Z_dem = create_synthetic_dem(size=300)

print(f"ขนาด DEM  : {Z_dem.shape}")
print(f"ค่าต่ำสุด : {Z_dem.min():.1f} m")
print(f"ค่าสูงสุด : {Z_dem.max():.1f} m")
print(f"ค่าเฉลี่ย : {Z_dem.mean():.1f} m")

# แสดง DEM แบบ 2D ก่อน
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(Z_dem, cmap='terrain', origin='upper')
plt.colorbar(im, ax=ax, label='Elevation (m)')
ax.set_title('DEM ภาคเหนือไทย (Synthetic) — 2D View', fontsize=13)
ax.set_xlabel('Column (pixel)')
ax.set_ylabel('Row (pixel)')
plt.tight_layout()
plt.show()

---
## 🔵 Level 1 — matplotlib 3D Surface

> **เป้าหมาย**: เข้าใจ concept ของ 3D surface plot, ปรับมุมกล้อง, เพิ่ม colormap

### วิธีทำงาน:
```
DEM array (2D)  →  meshgrid (X, Y)  →  plot_surface(X, Y, Z)
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 1A: 3D Surface พื้นฐาน
# ─────────────────────────────────────────────────────────────

# Downsample เพื่อให้ plot เร็ว (ทุก 3 pixel)
step = 3
Z_small = Z_dem[::step, ::step]

# สร้าง coordinate grid
ny, nx = Z_small.shape
x_arr = np.arange(nx)
y_arr = np.arange(ny)
X_grid, Y_grid = np.meshgrid(x_arr, y_arr)

# สร้าง figure
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot surface
surf = ax.plot_surface(
    X_grid, Y_grid, Z_small,
    cmap='terrain',     # colormap สำหรับ terrain
    alpha=0.9,          # ความโปร่งใส
    linewidth=0,        # ไม่แสดง edge lines
    antialiased=True
)

# Color bar
fig.colorbar(surf, ax=ax, shrink=0.4, label='Elevation (m)')

# ตั้งค่า labels
ax.set_xlabel('Easting (pixel)')
ax.set_ylabel('Northing (pixel)')
ax.set_zlabel('Elevation (m)')
ax.set_title('3D DEM — ภาคเหนือไทย\n(matplotlib)', fontsize=13)

# ปรับมุมกล้อง  ← ลองเปลี่ยนค่านี้!
ax.view_init(elev=30, azim=225)

plt.tight_layout()
plt.savefig('dem_3d_matplotlib.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 บันทึก: dem_3d_matplotlib.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 1B: เปรียบ 4 มุมกล้อง + เพิ่ม Contour
# ─────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(16, 12))
views = [
    (30, 45,  'มุม NE — elev=30, azim=45'),
    (30, 225, 'มุม SW — elev=30, azim=225'),
    (60, 135, 'มุม Bird-eye — elev=60, azim=135'),
    (15, 0,   'มุม Profile — elev=15, azim=0'),
]

for i, (elev, azim, title) in enumerate(views):
    ax = fig.add_subplot(2, 2, i+1, projection='3d')

    ax.plot_surface(X_grid, Y_grid, Z_small, cmap='terrain', alpha=0.85, linewidth=0)

    # เพิ่ม Contour ที่ระนาบล่าง
    ax.contour(X_grid, Y_grid, Z_small,
               zdir='z', offset=Z_small.min()-50,
               cmap='Blues_r', alpha=0.4, levels=10)

    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=10)
    ax.set_zlim(Z_small.min()-50, Z_small.max())
    ax.tick_params(labelsize=7)

plt.suptitle('เปรียบเทียบมุมมอง 3D DEM (4 มุม)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('dem_3d_views.png', dpi=120, bbox_inches='tight')
plt.show()

# 💡 ลองเอง: เปลี่ยน cmap เป็น 'gist_earth', 'copper', หรือ 'twilight'

### 📝 แบบฝึกหัด Level 1
1. เปลี่ยน `cmap` เป็น `'gist_earth'` แล้ว plot ใหม่
2. เปลี่ยน `elev` และ `azim` เพื่อหามุมที่ดูดีที่สุด
3. เพิ่ม `ax.set_zlim(200, 2600)` แล้วสังเกตผลลัพธ์

---
## 🟢 Level 2 — Plotly Interactive 3D Surface

> **เป้าหมาย**: สร้าง 3D map ที่ **หมุน/zoom ได้** และ export เป็น HTML

### ข้อดีของ Plotly:
- 🖱️ หมุน/zoom ด้วย mouse
- 🏷️ Hover แสดง elevation ได้
- 💾 Export HTML ส่งต่อได้โดยไม่ต้อง install Python

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 2A: Plotly Surface พื้นฐาน
# ─────────────────────────────────────────────────────────────

# Downsample สำหรับ plotly (ทุก 2 pixel)
Z_plotly = Z_dem[::2, ::2]

# สร้าง Figure
fig = go.Figure(data=[
    go.Surface(
        z=Z_plotly,
        colorscale='Earth',       # ลองเปลี่ยนเป็น 'Viridis', 'Plasma', 'Hot'
        showscale=True,
        colorbar=dict(
            title='Elevation (m)',
            titleside='right'
        ),
        lighting=dict(
            ambient=0.6,
            diffuse=0.8,
            roughness=0.9,
            specular=0.2
        )
    )
])

# Layout
fig.update_layout(
    title=dict(
        text='3D DEM ภาคเหนือไทย — Plotly Interactive',
        x=0.5, font=dict(size=16)
    ),
    width=900, height=600,
    scene=dict(
        xaxis_title='X (pixel)',
        yaxis_title='Y (pixel)',
        zaxis_title='Elevation (m)',
        aspectratio=dict(x=1, y=1, z=0.4),   # z_scale ← ปรับความ "สูง" ของเขา
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.0)
        )
    ),
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()
print("🖱️  ลองหมุนแผนที่โดยการคลิกแล้วลาก!")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 2B: เพิ่ม Annotation จุดที่สำคัญ
# ─────────────────────────────────────────────────────────────

# จุดที่จะ mark (row, col ใน array ที่ downsample)
# หา peak สูงสุด
peak_idx = np.unravel_index(np.argmax(Z_plotly), Z_plotly.shape)
peak_row, peak_col = peak_idx
peak_elev = Z_plotly[peak_row, peak_col]

# จุดตัวอย่างอื่น
ny2, nx2 = Z_plotly.shape
center_row, center_col = ny2//2, nx2//2

fig2 = go.Figure()

# Surface
fig2.add_trace(go.Surface(
    z=Z_plotly, colorscale='Earth',
    showscale=True, opacity=0.95,
    colorbar=dict(title='Elevation (m)', len=0.6)
))

# Annotation: Peak
fig2.add_trace(go.Scatter3d(
    x=[peak_col], y=[peak_row], z=[peak_elev + 80],
    mode='markers+text',
    marker=dict(size=8, color='red', symbol='diamond'),
    text=[f'Peak: {peak_elev:.0f} m'],
    textposition='top center',
    name='ยอดเขาสูงสุด'
))

# Annotation: Center valley
valley_elev = Z_plotly[center_row, center_col]
fig2.add_trace(go.Scatter3d(
    x=[center_col], y=[center_row], z=[valley_elev + 50],
    mode='markers+text',
    marker=dict(size=6, color='blue', symbol='circle'),
    text=[f'Valley: {valley_elev:.0f} m'],
    textposition='top center',
    name='หุบเขา'
))

fig2.update_layout(
    title='3D DEM พร้อม Annotation',
    width=900, height=600,
    scene=dict(
        aspectratio=dict(x=1, y=1, z=0.4),
        zaxis_title='Elevation (m)'
    ),
    showlegend=True
)

fig2.show()
print(f"🏔️  Peak elevation: {peak_elev:.1f} m ที่ pixel ({peak_col}, {peak_row})")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 2C: Export เป็น HTML
# ─────────────────────────────────────────────────────────────

fig2.write_html('dem_3d_plotly.html')
print("💾 บันทึก: dem_3d_plotly.html")
print("   → ดาวน์โหลดจาก Colab แล้วเปิดใน browser ได้เลย!")
print("   → ส่งไฟล์ให้คนอื่นดูได้โดยไม่ต้อง install Python")

# ดาวน์โหลดจาก Colab อัตโนมัติ
try:
    from google.colab import files
    files.download('dem_3d_plotly.html')
except ImportError:
    print("   (ไม่ได้รันบน Colab — ไฟล์อยู่ใน current directory)")

### 📝 แบบฝึกหัด Level 2
1. เปลี่ยน `colorscale` เป็น `'Viridis'`, `'Hot'`, `'RdBu'` แล้วดูผล
2. เปลี่ยน `aspectratio z` จาก `0.4` เป็น `1.0` — ผลต่างกันอย่างไร?
3. เพิ่ม `contours` บน surface โดยเพิ่มใน `go.Surface()`: `contours=dict(z=dict(show=True, usecolormap=True, width=1))`

---
## 🟣 Level 3 — PyDeck: Geospatial 3D บนแผนที่จริง

> **เป้าหมาย**: แสดงข้อมูล geospatial 3D บน basemap จริง เช่น PM2.5, จุดตัวอย่าง

PyDeck ใช้ **deck.gl** ซึ่งเป็น WebGL library ของ Uber  
เหมาะสำหรับข้อมูล point/polygon/raster ที่มี **좌표จริง (lat/lon)**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 3A: สร้างข้อมูล PM2.5 จำลอง 17 จังหวัดภาคเหนือ
# ─────────────────────────────────────────────────────────────

# ข้อมูลจังหวัดภาคเหนือ (lat/lon, PM2.5 ตัวอย่าง)
north_provinces = {
    'Province': [
        'เชียงใหม่', 'เชียงราย', 'น่าน', 'แพร่', 'ลำปาง',
        'ลำพูน', 'แม่ฮ่องสอน', 'พะเยา', 'อุตรดิตถ์', 'สุโขทัย',
        'กำแพงเพชร', 'พิษณุโลก', 'พิจิตร', 'เพชรบูรณ์', 'ตาก',
        'นครสวรรค์', 'อุทัยธานี'
    ],
    'lat': [
        18.787, 19.910, 18.783, 18.147, 18.291,
        18.574, 19.301, 19.143, 17.623, 17.010,
        16.483, 16.821, 16.443, 16.435, 16.870,
        15.703, 15.380
    ],
    'lon': [
        98.992, 99.833, 100.773, 100.130, 99.493,
        99.007, 97.967, 99.893, 100.093, 99.821,
        99.522, 100.266, 100.111, 101.158, 99.127,
        100.137, 100.024
    ],
    'pm25': [
        # ค่า PM2.5 μg/m³ จำลอง (ฤดูหมอกควัน)
        65.2, 58.4, 45.1, 72.3, 61.8,
        55.6, 48.9, 52.7, 38.4, 42.1,
        35.6, 88.5, 41.2, 39.8, 53.4,   # พิษณุโลก outlier!
        33.2, 31.5
    ]
}

df_pm25 = pd.DataFrame(north_provinces)

# กำหนดสีตามระดับ PM2.5 (RGB)
def pm25_color(val):
    if val <= 25:   return [0, 228, 0]      # ดี (เขียว)
    elif val <= 50: return [255, 255, 0]    # ปานกลาง (เหลือง)
    elif val <= 75: return [255, 126, 0]    # ไม่ดีสำหรับกลุ่มเสี่ยง (ส้ม)
    elif val <= 100:return [255, 0, 0]      # ไม่ดี (แดง)
    else:           return [143, 63, 151]   # อันตราย (ม่วง)

df_pm25['color'] = df_pm25['pm25'].apply(pm25_color)
df_pm25['radius'] = 15000  # ขนาดวงกลม (เมตร)

print(df_pm25[['Province','pm25']].to_string(index=False))
print(f"\nค่า PM2.5 สูงสุด: {df_pm25['pm25'].max():.1f} ({df_pm25.loc[df_pm25['pm25'].idxmax(), 'Province']})")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 3B: ColumnLayer — แสดง PM2.5 เป็นเสา 3D
# ─────────────────────────────────────────────────────────────

# ColumnLayer: แสดงค่าเป็นเสาสูงตามค่า PM2.5
column_layer = pdk.Layer(
    'ColumnLayer',
    data=df_pm25,
    get_position=['lon', 'lat'],
    get_elevation='pm25',
    elevation_scale=800,          # scale factor: pm25 μg → ความสูงบนแผนที่
    radius=20000,                 # รัศมีเสา (เมตร)
    get_fill_color='color',
    pickable=True,
    auto_highlight=True,
    coverage=0.8
)

# ScatterplotLayer: วงกลมแสดงตำแหน่ง
scatter_layer = pdk.Layer(
    'ScatterplotLayer',
    data=df_pm25,
    get_position=['lon', 'lat'],
    get_radius=8000,
    get_fill_color=[255, 255, 255, 100],
    get_line_color=[0, 0, 0],
    line_width_min_pixels=1
)

# ViewState: จุดศูนย์กลางและมุมมอง
view_state = pdk.ViewState(
    latitude=18.0,
    longitude=99.5,
    zoom=6.5,
    pitch=50,       # มุมเงยกล้อง (0=overhead, 90=horizontal)
    bearing=0       # หมุนแผนที่ (0=North up)
)

# Tooltip
tooltip = {
    'html': '<b>{Province}</b><br/>PM2.5: {pm25} μg/m³',
    'style': {'background': 'steelblue', 'color': 'white', 'fontSize': '14px', 'padding': '8px'}
}

r = pdk.Deck(
    layers=[column_layer, scatter_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style='mapbox://styles/mapbox/dark-v10'  # dark basemap
)

r.to_html('pm25_3d_column.html')
print("💾 บันทึก: pm25_3d_column.html")

# แสดงใน Colab
r

In [ ]:
# ─────────────────────────────────────────────────────────────
# Level 3C: HexagonLayer — Aggregation 3D
# (จำลอง hotspot data จากจุดตัวอย่างจำนวนมาก)
# ─────────────────────────────────────────────────────────────

np.random.seed(42)

# สร้าง fire hotspot จำลอง ภาคเหนือ
n_points = 500
fire_lats = np.random.uniform(17.5, 20.5, n_points)
fire_lons = np.random.uniform(97.5, 101.5, n_points)

# เพิ่ม cluster ใน Chiang Rai และ Mae Hong Son
cluster1_lat = np.random.normal(19.9, 0.5, 150)
cluster1_lon = np.random.normal(99.8, 0.5, 150)
cluster2_lat = np.random.normal(19.3, 0.4, 120)
cluster2_lon = np.random.normal(98.0, 0.4, 120)

all_lats = np.concatenate([fire_lats, cluster1_lat, cluster2_lat])
all_lons = np.concatenate([fire_lons, cluster1_lon, cluster2_lon])

df_fire = pd.DataFrame({'lat': all_lats, 'lon': all_lons})

hexagon_layer = pdk.Layer(
    'HexagonLayer',
    data=df_fire,
    get_position=['lon', 'lat'],
    auto_highlight=True,
    elevation_scale=200,
    pickable=True,
    elevation_range=[0, 2000],
    extruded=True,              # ทำให้เป็น 3D
    coverage=0.9,
    radius=25000,               # ขนาด hexagon (เมตร)
    color_range=[
        [254, 229, 217],
        [252, 174, 145],
        [251, 106, 74],
        [222, 45, 38],
        [165, 15, 21],
        [103, 0, 13]
    ]
)

view_fire = pdk.ViewState(
    latitude=19.0, longitude=99.0,
    zoom=6.0, pitch=55, bearing=20
)

r_fire = pdk.Deck(
    layers=[hexagon_layer],
    initial_view_state=view_fire,
    tooltip={'text': 'Fire hotspot count: {elevationValue}'},
    map_style='mapbox://styles/mapbox/satellite-v9'
)

r_fire.to_html('fire_hotspot_3d.html')
print("💾 บันทึก: fire_hotspot_3d.html")
print(f"🔥 จำนวน fire hotspot จำลอง: {len(df_fire)} จุด")

r_fire

### 📝 แบบฝึกหัด Level 3
1. เปลี่ยน `elevation_scale` ใน ColumnLayer จาก `800` เป็น `2000` — ผลต่างกันอย่างไร?
2. เปลี่ยน `pitch` จาก `50` เป็น `70` — มุมมองเปลี่ยนไปอย่างไร?
3. ลอง `map_style='mapbox://styles/mapbox/light-v10'` แทน dark theme

---
## 🎓 สรุป — เลือก Tool ตามงาน

| Tool | ใช้เมื่อ | ข้อดี | ข้อจำกัด |
|------|---------|-------|----------|
| **matplotlib** | Quick figure สำหรับ paper | ไม่ต้อง install เพิ่ม | ไม่ interactive |
| **Plotly** | Presentation / Dashboard | หมุนได้, export HTML | ต้อง install plotly |
| **PyDeck** | Geospatial storytelling | บนแผนที่จริง, สวย | ต้องมี MapBox key |

---
## 🚀 Next Step
- **Drape satellite image บน DEM**: ใช้ Plotly Heatmap overlay
- **Real DEM จาก GEE**: Export SRTM จาก Google Earth Engine → GeoTIFF → plot
- **Point Cloud 3D**: ใช้ `open3d` สำหรับ LiDAR data
- **Kepler.gl**: ทางเลือกของ PyDeck ที่มี GUI drag-and-drop

In [ ]:
# ─────────────────────────────────────────────────────────────
# BONUS: ดาวน์โหลดไฟล์ทั้งหมดจาก Colab
# ─────────────────────────────────────────────────────────────
output_files = [
    'dem_3d_matplotlib.png',
    'dem_3d_views.png',
    'dem_3d_plotly.html',
    'pm25_3d_column.html',
    'fire_hotspot_3d.html',
]

try:
    from google.colab import files
    for f in output_files:
        try:
            files.download(f)
            print(f"📥 {f}")
        except Exception:
            print(f"⚠️  {f} — ไม่พบไฟล์ (รัน cell นั้นก่อน)")
except ImportError:
    import os
    print("ไฟล์ output ทั้งหมด:")
    for f in output_files:
        exists = '✅' if os.path.exists(f) else '❌'
        print(f"  {exists}  {f}")